# keyboard.actions

> Keyboard navigation focus zone and action factories for the virtual collection.

In [ ]:
#| default_exp keyboard.actions

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from typing import Dict, Optional, Tuple

from cjm_fasthtml_keyboard_navigation.core.focus_zone import FocusZone
from cjm_fasthtml_keyboard_navigation.core.actions import KeyAction
from cjm_fasthtml_keyboard_navigation.core.navigation import ScrollOnly
from cjm_fasthtml_keyboard_navigation.components.system import KeyboardSystem

from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds
from cjm_fasthtml_virtual_collection.core.button_ids import VirtualCollectionButtonIds
from cjm_fasthtml_virtual_collection.core.models import VirtualCollectionUrls

## Focus Zone

In [ ]:
#| export
def create_collection_focus_zone(
    ids: VirtualCollectionHtmlIds,  # HTML IDs for this collection instance
    hidden_input_prefix: Optional[str] = None,  # Prefix for keyboard nav hidden inputs
) -> FocusZone:  # Configured focus zone for the collection
    """Create a focus zone for a virtual collection viewport."""
    return FocusZone(
        id=ids.collection,
        item_selector=None,
        navigation=ScrollOnly(),
        zone_focus_classes=(),
        item_focus_classes=(),
        hidden_input_prefix=hidden_input_prefix or f"{ids.prefix}-vc",
    )

## Navigation Actions

In [ ]:
#| export
def create_collection_nav_actions(
    zone_id: str,  # Focus zone ID to restrict actions to
    button_ids: VirtualCollectionButtonIds,  # Button IDs for HTMX triggers
    disable_in_modes: Tuple[str, ...] = (),  # Mode names that disable navigation
) -> Tuple[KeyAction, ...]:  # Standard collection navigation actions
    """Create standard keyboard navigation actions for a virtual collection."""
    zone_ids = (zone_id,)
    not_modes = disable_in_modes if disable_in_modes else ()

    return (
        # --- Row navigation ---
        KeyAction(
            key="ArrowUp",
            htmx_trigger=button_ids.nav_up,
            zone_ids=zone_ids,
            not_modes=not_modes,
            description="Previous row",
            hint_group="Navigation",
        ),
        KeyAction(
            key="ArrowDown",
            htmx_trigger=button_ids.nav_down,
            zone_ids=zone_ids,
            not_modes=not_modes,
            description="Next row",
            hint_group="Navigation",
        ),

        # --- Page jump ---
        KeyAction(
            key="PageUp",
            htmx_trigger=button_ids.nav_page_up,
            zone_ids=zone_ids,
            not_modes=not_modes,
            description="Page up",
            hint_group="Navigation",
        ),
        KeyAction(
            key="PageDown",
            htmx_trigger=button_ids.nav_page_down,
            zone_ids=zone_ids,
            not_modes=not_modes,
            description="Page down",
            hint_group="Navigation",
        ),

        # --- First/last ---
        KeyAction(
            key="Home",
            htmx_trigger=button_ids.nav_first,
            zone_ids=zone_ids,
            not_modes=not_modes,
            description="First row",
            hint_group="Navigation",
        ),
        KeyAction(
            key="End",
            htmx_trigger=button_ids.nav_last,
            zone_ids=zone_ids,
            not_modes=not_modes,
            description="Last row",
            hint_group="Navigation",
        ),

        # --- Activate focused row ---
        KeyAction(
            key="Enter",
            htmx_trigger=button_ids.activate,
            zone_ids=zone_ids,
            not_modes=not_modes,
            description="Activate row",
            hint_group="Actions",
        ),
        KeyAction(
            key=" ",
            htmx_trigger=button_ids.activate,
            zone_ids=zone_ids,
            not_modes=not_modes,
            description="Activate row",
            hint_group="Actions",
            show_in_hints=False,
        ),
    )

## URL Map Builder

In [ ]:
#| export
def build_collection_url_map(
    button_ids: VirtualCollectionButtonIds,  # Button IDs for this collection
    urls: VirtualCollectionUrls,  # URL bundle for routing
) -> Dict[str, str]:  # Mapping of button ID -> route URL
    """Build url_map for render_keyboard_system with all collection buttons."""
    return {
        button_ids.nav_up: urls.nav_up,
        button_ids.nav_down: urls.nav_down,
        button_ids.nav_page_up: urls.nav_page_up,
        button_ids.nav_page_down: urls.nav_page_down,
        button_ids.nav_first: urls.nav_first,
        button_ids.nav_last: urls.nav_last,
        button_ids.activate: urls.activate,
    }

## Nav Button Sync

In [ ]:
#| export
def apply_nav_sync(
    kb_system: KeyboardSystem,         # Rendered keyboard system to patch
    ids: VirtualCollectionHtmlIds,     # HTML IDs for this collection
) -> None:  # Modifies kb_system.action_buttons in place
    """Add hx-sync to nav buttons so rapid input aborts stale in-flight requests.

    Prevents race conditions between cursor-only and full-window OOB responses
    during rapid keyboard or scroll wheel navigation.
    """
    sync_target = f"#{ids.collection}:abort"
    nav_prefix = f"{ids.prefix}-btn-nav-"
    for child in kb_system.action_buttons.children:
        if hasattr(child, 'attrs') and child.attrs.get('id', '').startswith(nav_prefix):
            child.attrs['hx-sync'] = sync_target

## Tests

In [ ]:
from cjm_fasthtml_virtual_collection.core.html_ids import VirtualCollectionHtmlIds
from cjm_fasthtml_virtual_collection.core.button_ids import VirtualCollectionButtonIds
from cjm_fasthtml_virtual_collection.core.models import VirtualCollectionUrls

ids = VirtualCollectionHtmlIds(prefix="t")
btn_ids = VirtualCollectionButtonIds(prefix="t")
urls = VirtualCollectionUrls(
    nav_up="/nav_up", nav_down="/nav_down",
    nav_page_up="/nav_page_up", nav_page_down="/nav_page_down",
    nav_first="/nav_first", nav_last="/nav_last",
    activate="/activate",
)

In [ ]:
# Test focus zone
zone = create_collection_focus_zone(ids)
assert zone.id == "t-collection"
assert zone.item_selector is None
assert zone.zone_focus_classes == ()
print(f"Zone: id={zone.id}, nav={zone.navigation}")

Zone: id=t-collection, nav=ScrollOnly()


In [ ]:
# Test nav actions
actions = create_collection_nav_actions(zone.id, btn_ids)
assert len(actions) == 8  # 6 nav + Enter + Space

keys = [a.key for a in actions]
assert "ArrowUp" in keys
assert "ArrowDown" in keys
assert "PageUp" in keys
assert "PageDown" in keys
assert "Home" in keys
assert "End" in keys
assert "Enter" in keys
assert " " in keys

# All actions should use htmx_trigger (not js_callback)
for action in actions:
    assert action.htmx_trigger is not None
    assert action.js_callback is None

# Enter and Space target activate button
enter_action = [a for a in actions if a.key == "Enter"][0]
space_action = [a for a in actions if a.key == " "][0]
assert enter_action.htmx_trigger == btn_ids.activate
assert space_action.htmx_trigger == btn_ids.activate
assert space_action.show_in_hints == False

print(f"Nav actions: {len(actions)} actions, all htmx_trigger")
for a in actions:
    print(f"  {a.key!r} -> {a.htmx_trigger}")

In [ ]:
# Test URL map
url_map = build_collection_url_map(btn_ids, urls)
assert len(url_map) == 7  # 6 nav + activate
assert url_map[btn_ids.nav_up] == "/nav_up"
assert url_map[btn_ids.nav_down] == "/nav_down"
assert url_map[btn_ids.nav_first] == "/nav_first"
assert url_map[btn_ids.nav_last] == "/nav_last"
assert url_map[btn_ids.activate] == "/activate"
print(f"URL map: {len(url_map)} entries")

In [ ]:
# Test apply_nav_sync
from fasthtml.common import to_xml
from cjm_fasthtml_keyboard_navigation.core.manager import ZoneManager
from cjm_fasthtml_keyboard_navigation.components.system import render_keyboard_system

zone = create_collection_focus_zone(ids)
actions = create_collection_nav_actions(zone.id, btn_ids)
manager = ZoneManager(zones=(zone,), actions=actions)
url_map = build_collection_url_map(btn_ids, urls)
target_map = {btn_id: "#rows" for btn_id in url_map}
swap_map = {btn_id: "none" for btn_id in url_map}

kb = render_keyboard_system(manager, url_map=url_map, target_map=target_map, swap_map=swap_map)
apply_nav_sync(kb, ids)

# Verify nav buttons have hx-sync, activate button does not
nav_prefix = f"{ids.prefix}-btn-nav-"
for child in kb.action_buttons.children:
    xml = to_xml(child)
    btn_id = child.attrs.get('id', '')
    if btn_id.startswith(nav_prefix):
        assert f'hx-sync="#{ids.collection}:abort"' in xml, f"Missing hx-sync on {btn_id}"
    else:
        assert 'hx-sync' not in xml, f"Unexpected hx-sync on {btn_id}"

print(f"apply_nav_sync: nav buttons have hx-sync, activate does not")

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()